# 🏈 NFL Pick Em — Monte Carlo Projections

Generates win probability snapshots for any week of the season.

**Run order:** Cell 1 → 2 → 3 → 4 → 5

- Pulls picks + actual wins from BigQuery
- Fetches live NFL team win % from ESPN API
- Runs 50,000 simulated seasons
- Writes results to the `projections` table in BigQuery

**To snapshot a different week:** change `AS_OF_WEEK` in Cell 2 and re-run cells 4 and 5.

In [ ]:
# ── CELL 1: Authenticate + install ───────────────────────────────
from google.colab import auth
auth.authenticate_user()

!pip install google-cloud-bigquery pyarrow requests --quiet

print('Ready!')

In [ ]:
# ── CELL 2: Config ────────────────────────────────────────────────
AS_OF_WEEK  = 10    # ← change to snapshot any week (1-18)
SIMULATIONS = 50000

PROJECT_ID  = 'nfl-pick-em-493506'
DATASET_ID  = 'nfl_pickem'
SEASON_YEAR = 2025
TOTAL_WEEKS = 18

print(f'Config set — generating projections as of Week {AS_OF_WEEK}')

In [ ]:
# ── CELL 3: Define all functions ──────────────────────────────────
import requests
import random
import pandas as pd
from datetime import datetime, timezone
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

def tbl(name):
    return f'{PROJECT_ID}.{DATASET_ID}.{name}'

def bq_load(df, table_name):
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
    job = client.load_table_from_dataframe(df, tbl(table_name), job_config=job_config)
    job.result()
    print(f'  ✅ {table_name}: {len(df)} rows loaded')

def fetch_espn_team_win_pct():
    """
    Fetches current NFL standings from ESPN.
    Returns dict of { 'Eagles': 0.800, 'Chiefs': 0.900, ... }
    Extracts nickname as the last word of displayName.
    Also prints a full mapping table so you can verify every team matched.
    """
    url = 'https://site.api.espn.com/apis/site/v2/sports/football/nfl/standings'
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    win_pct = {}
    for group in data.get('children', []):
        for entry in group.get('standings', {}).get('entries', []):
            full_name = entry['team']['displayName']       # e.g. 'Philadelphia Eagles'
            nickname  = full_name.split()[-1]              # e.g. 'Eagles'
            stats = {s['name']: s['value'] for s in entry.get('stats', [])}
            wins   = stats.get('wins',   0)
            losses = stats.get('losses', 0)
            ties   = stats.get('ties',   0)
            games  = wins + losses + ties
            pct    = round(wins / games, 3) if games > 0 else 0.500
            win_pct[nickname] = pct

    print(f'ESPN standings fetched: {len(win_pct)} teams')
    print(f'{"Team":<15} {"Win %":>6}')
    print('-' * 23)
    for team, pct in sorted(win_pct.items()):
        print(f'{team:<15} {pct:>6.3f}')
    return win_pct

def fetch_picks_from_bq(season_year, as_of_week):
    """
    Pulls all picks from BigQuery.
    Splits into done_picks (weeks <= as_of_week) and future_picks (weeks > as_of_week).
    """
    query = f"""
    SELECT
        pk.player_id,
        p.display_name,
        pk.week_number,
        pk.team_picked
    FROM `{PROJECT_ID}.{DATASET_ID}.picks` pk
    JOIN `{PROJECT_ID}.{DATASET_ID}.players` p
        ON  pk.player_id   = p.player_id
        AND pk.season_year = p.season_year
    WHERE pk.season_year = {season_year}
      AND pk.team_picked IS NOT NULL
    ORDER BY pk.player_id, pk.week_number
    """
    rows = client.query(query).result()
    players = {}
    for row in rows:
        pid = row.player_id
        if pid not in players:
            players[pid] = {
                'player_id':    pid,
                'display_name': row.display_name,
                'done_picks':   [],
                'future_picks': [],
            }
        if row.week_number <= as_of_week:
            players[pid]['done_picks'].append(row.team_picked)
        else:
            players[pid]['future_picks'].append(row.team_picked)
    print(f'BigQuery picks: {len(players)} players fetched')
    return players

def verify_team_mapping(players_data, win_pct):
    """
    Checks every team name in picks against ESPN win_pct dict.
    Prints any mismatches so you can catch mapping errors before running simulation.
    """
    all_picked = set()
    for p in players_data.values():
        all_picked.update(p['done_picks'])
        all_picked.update(p['future_picks'])

    unmatched = [t for t in sorted(all_picked) if t not in win_pct]
    matched   = [t for t in sorted(all_picked) if t in win_pct]

    print(f'\nTeam mapping check:')
    print(f'  Matched   : {len(matched)} teams')
    print(f'  Unmatched : {len(unmatched)} teams')
    if unmatched:
        print(f'  ⚠️  Unmatched team names (will use 0.500 fallback):')
        for t in unmatched:
            print(f'     - "{t}"')
    else:
        print(f'  ✅ All team names matched successfully')

def fetch_wins_to_date(season_year, as_of_week):
    """
    Returns dict of { player_id: actual_wins } through as_of_week.
    Requires weekly_results to be loaded for weeks 1..as_of_week.
    """
    query = f"""
    SELECT
        pk.player_id,
        COUNT(*) AS wins
    FROM `{PROJECT_ID}.{DATASET_ID}.picks` pk
    JOIN `{PROJECT_ID}.{DATASET_ID}.weekly_results` wr
        ON  pk.season_year = wr.season_year
        AND pk.week_number = wr.week_number
        AND pk.team_picked = wr.team_won
    WHERE pk.season_year = {season_year}
      AND pk.week_number <= {as_of_week}
    GROUP BY pk.player_id
    """
    rows = client.query(query).result()
    wins = {row.player_id: row.wins for row in rows}
    print(f'Wins to date: {len(wins)} players with at least 1 win through week {as_of_week}')
    return wins

def run_simulation(players_data, wins_to_date, win_pct, as_of_week, n=50000):
    """
    Monte Carlo simulation — 50,000 seasons simulated from as_of_week forward.
    Each future pick is a weighted coin flip using that team's current ESPN win %.
    Tiebreaker: lower avg NFL win % of all picks wins (rewards underdogs).
    Returns dict of { player_id: win_probability_pct }
    """
    random.seed(42)
    player_list  = list(players_data.values())
    win_counts   = {p['player_id']: 0 for p in player_list}
    avg_fallback = 0.450  # probability for unsubmitted weeks

    for i in range(n):
        sim = []
        for p in player_list:
            total = wins_to_date.get(p['player_id'], 0)

            for team in p['future_picks']:
                if random.random() < win_pct.get(team, 0.500):
                    total += 1

            submitted   = len(p['done_picks']) + len(p['future_picks'])
            unsubmitted = max(0, TOTAL_WEEKS - submitted)
            for _ in range(unsubmitted):
                if random.random() < avg_fallback:
                    total += 1

            all_picks = p['done_picks'] + p['future_picks']
            all_wps   = [win_pct.get(t, 0.500) for t in all_picks]
            avg_tb    = sum(all_wps) / len(all_wps) if all_wps else 0.500

            sim.append((p['player_id'], total, avg_tb))

        sim.sort(key=lambda x: (-x[1], x[2]))
        win_counts[sim[0][0]] += 1

        if (i + 1) % 10000 == 0:
            print(f'  Progress: {i+1:,} / {n:,}')

    return {pid: round(count / n * 100, 2) for pid, count in win_counts.items()}

def build_projection_rows(players_data, wins_to_date, win_pct,
                          win_probs, season_year, as_of_week, n_sims):
    rows = []
    now  = datetime.now(timezone.utc)
    for p in players_data.values():
        pid       = p['player_id']
        done      = p['done_picks']
        future    = p['future_picks']
        all_picks = done + future
        exp_future = sum(win_pct.get(t, 0.500) for t in future)
        all_wps    = [win_pct.get(t, 0.500) for t in all_picks]
        avg_tb     = round(sum(all_wps) / len(all_wps), 4) if all_wps else 0.500
        submitted  = len(all_picks)
        unsub      = max(0, TOTAL_WEEKS - submitted)
        wins_done  = wins_to_date.get(pid, 0)
        rows.append({
            'season_year':          season_year,
            'as_of_week':           as_of_week,
            'player_id':            pid,
            'display_name':         p['display_name'],
            'wins_to_date':         wins_done,
            'expected_future_wins': round(exp_future, 3),
            'projected_total':      round(wins_done + exp_future, 3),
            'avg_team_win_pct':     avg_tb,
            'win_probability':      win_probs.get(pid, 0.0),
            'weeks_remaining':      max(0, TOTAL_WEEKS - as_of_week),
            'unsubmitted_weeks':    unsub,
            'simulations_run':      n_sims,
            'created_at':           now,
        })
    return rows

print('Functions defined!')

In [ ]:
# ── CELL 4: Fetch data and verify team mapping ────────────────────
# Run this first to confirm ESPN team names match your picks
# before running the simulation. Fix any mismatches shown here.

print('Step 1/3 — Fetching ESPN team win %...')
team_win_pct = fetch_espn_team_win_pct()

print('\nStep 2/3 — Fetching picks from BigQuery...')
players_data = fetch_picks_from_bq(SEASON_YEAR, AS_OF_WEEK)

print('\nStep 3/3 — Verifying team name mapping...')
verify_team_mapping(players_data, team_win_pct)

print('\nIf all teams matched, proceed to Cell 5.')
print('If any unmatched teams are shown, check ESPN displayName vs your pick spelling.')

In [ ]:
# ── CELL 5: Run simulation and load to BigQuery ───────────────────
#
# NOTE: If re-running the same AS_OF_WEEK, drop and recreate the
# projections table in BigQuery console first to avoid duplicates:
#
#   DROP TABLE `nfl-pick-em-493506.nfl_pickem.projections`;
#   CREATE TABLE IF NOT EXISTS `nfl-pick-em-493506.nfl_pickem.projections` (...)
#
# Each unique AS_OF_WEEK value is safe to append without clearing.
# ─────────────────────────────────────────────────────────────────

print(f'Fetching wins to date from BigQuery...')
wins_to_date = fetch_wins_to_date(SEASON_YEAR, AS_OF_WEEK)

print(f'\nRunning Monte Carlo ({SIMULATIONS:,} simulations)...')
win_probs = run_simulation(
    players_data, wins_to_date, team_win_pct, AS_OF_WEEK, SIMULATIONS
)

print('\nBuilding output rows...')
proj_rows = build_projection_rows(
    players_data, wins_to_date, team_win_pct,
    win_probs, SEASON_YEAR, AS_OF_WEEK, SIMULATIONS
)

print(f'\nLoading {len(proj_rows)} rows into BigQuery...')
bq_load(pd.DataFrame(proj_rows), 'projections')

print(f'\n✅ Done! Week {AS_OF_WEEK} snapshot saved.')
print(f'\nTop 10 — Season {SEASON_YEAR} as of Week {AS_OF_WEEK}:')
print(f'{"Rank":<5} {"Player":<35} {"Wins":>5} {"Proj":>7} {"AvgTeamW%":>10} {"WinProb":>8}')
print('-' * 75)
top = sorted(proj_rows, key=lambda x: (-x['win_probability'], x['avg_team_win_pct']))
for i, r in enumerate(top[:10], 1):
    print(f"{i:<5} {r['display_name']:<35} "
          f"{r['wins_to_date']:>5} "
          f"{r['projected_total']:>7.1f} "
          f"{r['avg_team_win_pct']*100:>9.1f}% "
          f"{r['win_probability']:>7.1f}%")

In [ ]:
# ── CELL 6: Compare two snapshots ────────────────────────────────
# Shows how win probabilities shifted between any two weeks.
# Requires both weeks to already be loaded in the projections table.

COMPARE_WEEK_A = 8   # ← earlier week
COMPARE_WEEK_B = 10  # ← later week

query = f"""
SELECT
    a.display_name,
    a.wins_to_date                                       AS wins_wk{COMPARE_WEEK_A},
    ROUND(a.win_probability, 2)                          AS prob_wk{COMPARE_WEEK_A},
    b.wins_to_date                                       AS wins_wk{COMPARE_WEEK_B},
    ROUND(b.win_probability, 2)                          AS prob_wk{COMPARE_WEEK_B},
    ROUND(b.win_probability - a.win_probability, 2)      AS prob_change
FROM `{PROJECT_ID}.{DATASET_ID}.projections` a
JOIN `{PROJECT_ID}.{DATASET_ID}.projections` b
    ON  a.player_id   = b.player_id
    AND a.season_year = b.season_year
WHERE a.season_year  = {SEASON_YEAR}
  AND a.as_of_week   = {COMPARE_WEEK_A}
  AND b.as_of_week   = {COMPARE_WEEK_B}
ORDER BY prob_change DESC
"""

df_compare = client.query(query).to_dataframe()
print(f'Probability changes: Week {COMPARE_WEEK_A} → Week {COMPARE_WEEK_B}\n')
df_compare